# 01.3 — Correlation & Multicollinearity Analysis

## 1. Mục tiêu & Ranh giới Kiến trúc Khoa học

Notebook này thực hiện phân tích cấu trúc tương quan và đa cộng tuyến theo 3 trục chính trên tập **Dev Train** ($N = 6{,}329$ ngày, từ 2001-01-01 đến 2018-04-30):

```text
01.3 Correlation & Multicollinearity Analysis
│
├── Trục A: Feature–Target Association (Tương quan Đặc trưng – Mục tiêu)
│      └── Spearman rho (chính, phi tuyến đơn điệu trên thứ hạng)
│      └── Pearson r (bổ trợ kiểm tra tính tuyến tính)
│      └── Định vị: Tương quan đồng thời (contemporaneous association), không suy diễn dự báo.
│
├── Trục B: Predictor–Predictor Redundancy (Độ dư thừa giữa các biến dự báo)
│      └── Ma trận Spearman tam giác dưới cho 32 biến khí tượng tuyến tính
│      └── Bảng sàng lọc cặp tương quan mạnh (|rho| >= 0.8, heuristic screening)
│      └── Chẩn đoán VIF & SVD Condition Number (xử lý nghiêm ngặt ma trận suy biến rank)
│
└── Trục C: Redundancy Structure (Cấu trúc phân nhóm & Chiều không gian)
       └── Phân cụm thứ bậc (Hierarchical Clustering: average linkage, dissimilarity = 1 - |rho_s|)
       └── PCA chẩn đoán: Tách bạch Explained Variance (dư thừa) vs PC-Target Association (|rho|)
```

### Nguyên tắc Bằng chứng & Chống Rò rỉ Dữ liệu (Zero-Leakage Integrity):
1. **Quần thể phân tích Dev Train ($N = 6{,}329$)**: Toàn bộ tính toán chạy trên Dev Train. Tập Validation ($N = 731$) và Final Test ($N = 2{,}191$) được bảo toàn 100% không đụng tới (`untouched holdouts`).
2. **Explicit Predictor Allowlist**: Áp dụng danh sách allowlist gồm đúng 32 biến khí tượng tuyến tính + 2 biến hướng gió. Fail-fast nếu thiếu biến; loại trừ tự động tọa độ, ID, ngày tháng.
3. **Cô lập biến góc hướng gió (Circular Variables)**: `Hướng gió 2m` và `Hướng gió 10m` có tính chất tuần hoàn ($0^\circ \equiv 360^\circ$), bị loại khỏi toàn bộ VIF, PCA, phân cụm thứ bậc. Chỉ phân tích sin/cos như chẩn đoán phân rã bổ trợ độc lập.
4. **Frozen Clustering Protocol**: Sử dụng dissimilarity $1 - |ho_s|$ và liên kết `average`. Dendrogram là kết quả khoa học chính; số cụm cắt là tóm tắt thăm dò (exploratory summary).
5. **Đa cộng tuyến là Chẩn đoán, KHÔNG tự ý loại bỏ đặc trưng**: EDA chỉ cung cấp bằng chứng định lượng. Quyết định chọn lọc hoặc loại biến thuộc về `02 Feature Engineering` theo từng họ mô hình cụ thể.


In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Reconfigure encoding
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

# Path resolution
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.evaluation.protocols import load_canonical_data
from src.eda.CorrelationAnalysis import CorrelationAnalyzer, CorrelationVisualizer

# 1. Nạp tập Dev Train chuẩn (Zero-Leakage standard: 2001-01-01 -> 2018-04-30, N = 6,329)
# Validation (2018-05-01 -> 2020-04-30, N = 731) và Final Test (2020-05-01 -> 2026-04-30, N = 2,191) hoàn toàn untouched.
dev_train, val_set, test_set = load_canonical_data(return_dev_train=True)
print(f"📊 Tập Dev Train phân tích: {len(dev_train):,} ngày ({dev_train['Ngày'].min().date()} → {dev_train['Ngày'].max().date()})")

# 2. Khởi tạo CorrelationAnalyzer với strict schema validation
analyzer = CorrelationAnalyzer(
    df=dev_train,
    target_col="Lượng mưa",
    date_col="Ngày",
    strict_schema=True
)

assert analyzer.schema_validation["status"] == "PASSED"
assert len(analyzer.linear_predictor_cols) == 32
assert len(analyzer.circular_cols) == 2
print("\n✅ Schema validation passed: Exactly 32 linear + 2 circular wind = 34 approved meteorological predictors.")


📊 Tập Dev Train phân tích: 6,329 ngày (2001-01-01 → 2018-04-30)
🔍 CORRELATION ANALYZER INITIALIZED
   📊 Analysis Population Rows: 6,329
   📅 Temporal Window: 2001-01-01 → 2018-04-30
   🎯 Target Variable: Lượng mưa
   🔢 Linear Meteorological Predictors: 32
   🔄 Circular Wind Variables (Isolated): 2

✅ Schema validation passed: Exactly 32 linear + 2 circular wind = 34 approved meteorological predictors.


In [2]:
# 1. Tính toán tương quan đồng thời (Contemporaneous Association) giữa đặc trưng tuyến tính và lượng mưa
df_target_assoc = analyzer.analyze_target_associations()

print("📋 BẢNG TƯƠNG QUAN ĐỒNG THỜI (CONTEMPORANEOUS MONOTONIC & LINEAR ASSOCIATION):")
print(df_target_assoc[["feature", "spearman_rho", "pearson_r", "abs_spearman"]].head(15).to_string(index=False))

print("\n💡 Ghi chú phương pháp luận quan trọng:")
print("   - Thước đo chính: Spearman rho (đơn điệu phi tuyến trên thứ hạng). Pearson r đóng vai trò bổ trợ.")
print("   - Định vị: Đo lường mối liên hệ đồng thời (contemporaneous association) tại cùng mốc thời gian t trong lịch sử.")
print("   - Ranh giới dự báo: KHÔNG đồng nghĩa với khả năng dự báo tại inference time hoặc tính sẵn có theo Future Covariate Contract.")

# 2. Trực quan hóa Top Contemporaneous Associations
viz = CorrelationVisualizer(analyzer)
fig_a = viz.visualize_target_associations(top_n=15, figsize=(12, 7), return_fig=True)
plt.show()


📋 BẢNG TƯƠNG QUAN ĐỒNG THỜI (CONTEMPORANEOUS MONOTONIC & LINEAR ASSOCIATION):
                    feature  spearman_rho  pearson_r  abs_spearman
           Biên độ nhiệt 2m     -0.764546  -0.448664      0.764546
         Độ ẩm tuyệt đối 2m      0.738019   0.401610      0.738019
              Điểm sương 2m      0.737136   0.383777      0.737136
         Độ ẩm tương đối 2m      0.733239   0.419507      0.733239
           Độ ẩm đất bề mặt      0.628751   0.429888      0.628751
Bức xạ trực tiếp pháp tuyến     -0.616417  -0.387486      0.616417
         Nhiệt độ tối đa 2m     -0.596989  -0.365889      0.596989
    Bức xạ sóng ngắn bề mặt     -0.518605  -0.377359      0.518605
          Độ ẩm đất vùng rễ      0.516035   0.304563      0.516035
      Bức xạ quang hợp tổng     -0.474569  -0.362500      0.474569
             Áp suất bề mặt     -0.470623  -0.261203      0.470623
      Bức xạ sóng dài xuống      0.451181   0.240619      0.451181
          Độ ẩm đất mặt cắt      0.446068   0.23829

In [3]:
# 1. Tương quan cặp đôi giữa các biến dự báo tuyến tính (Spearman Matrix)
pairwise_res = analyzer.analyze_pairwise_correlations(threshold=0.8)
strong_pairs = pairwise_res["strong_pairwise_associations"]

print(f"📊 TỔNG KẾT TƯƠNG QUAN CẶP ĐÔI ({pairwise_res['stats_summary']['n_pairs']} cặp biến tuyến tính):")
print(f"   • Mean |ρ|:   {pairwise_res['stats_summary']['mean_abs_rho']:.4f}")
print(f"   • Median |ρ|: {pairwise_res['stats_summary']['median_abs_rho']:.4f}")
print(f"   • Max |ρ|:    {pairwise_res['stats_summary']['max_abs_rho']:.4f}")

if not strong_pairs.empty:
    print(f"\n🔗 CÁC CẶP BIẾN CÓ LIÊN HỆ MẠNH (|ρ| >= 0.8, Heuristic Screening): {len(strong_pairs)} cặp")
    print(strong_pairs[["feature_1", "feature_2", "spearman_rho", "pearson_r"]].to_string(index=False))

# 2. Phân tích Đa cộng tuyến (VIF & SVD Matrix Conditioning)
vif_res = analyzer.analyze_multicollinearity()
mat_diag = vif_res["matrix_diagnostics"]

print("\n🔬 CHẨN ĐOÁN MA TRẬN ĐẶC TRƯNG TUYẾN TÍNH (SVD Conditioning trên X_scaled):")
print(f"   • Số đặc trưng:    {mat_diag['n_features']}")
print(f"   • Matrix Rank:     {mat_diag['matrix_rank']} (Rank Deficient: {mat_diag['rank_deficient']})")
print(f"   • Condition Num:   {mat_diag['condition_number']:.2e} (Near Singular: {mat_diag['near_singular']})")
print(f"   • Status:          {mat_diag['computation_status']}")

print("\n📋 BẢNG ĐIỂM VIF (TOP 10 CAO NHẤT):")
print(vif_res["vif_df"].head(10).to_string(index=False))

# 3. Trực quan hóa Heatmap & VIF
fig_b1 = viz.visualize_pairwise_heatmap(figsize=(14, 12), return_fig=True)
plt.show()

fig_b2 = viz.visualize_vif(top_n=15, figsize=(12, 7), return_fig=True)
plt.show()


📊 TỔNG KẾT TƯƠNG QUAN CẶP ĐÔI (496 cặp biến tuyến tính):
   • Mean |ρ|:   0.3866
   • Median |ρ|: 0.3494
   • Max |ρ|:    0.9997

🔗 CÁC CẶP BIẾN CÓ LIÊN HỆ MẠNH (|ρ| >= 0.8, Heuristic Screening): 50 cặp
                  feature_1                   feature_2  spearman_rho  pearson_r
              Điểm sương 2m          Độ ẩm tuyệt đối 2m        0.9997     0.9966
                 Bức xạ UVB                   Chỉ số UV        0.9964     0.9962
              Tốc độ gió 2m              Tốc độ gió 10m        0.9963     0.9958
    Bức xạ sóng ngắn bề mặt       Bức xạ quang hợp tổng        0.9953     0.9967
       Tốc độ gió tối đa 2m       Tốc độ gió tối đa 10m        0.9937     0.9946
          Độ ẩm đất vùng rễ           Độ ẩm đất mặt cắt        0.9904     0.9917
      Bức xạ quang hợp tổng                  Bức xạ UVA        0.9843     0.9889
Bức xạ sóng ngắn trời quang Bức xạ quang hợp trời quang        0.9786     0.9836
           Độ ẩm đất bề mặt           Độ ẩm đất vùng rễ        0.977

In [4]:
# 1. Phân cụm Thứ bậc (Hierarchical Clustering: average linkage, dissimilarity = 1 - |rho_s|)
clustering_res = analyzer.analyze_feature_clustering(requested_n_clusters=5)

print("🌳 PHÂN CỤM THỨ BẬC CÁC ĐẶC TRƯNG TUYẾN TÍNH (Average Linkage):")
print(f"   • Thước đo khác biệt:     {clustering_res['dissimilarity_metric']}")
print(f"   • Phương pháp liên kết:    {clustering_res['linkage_method']}")
print(f"   • Số cụm yêu cầu (Heuristic): {clustering_res['requested_n_clusters']}")
print(f"   • Số cụm thực tế (Cut):   {clustering_res['actual_n_clusters']}")
print("\n📦 DANH SÁCH CÁC CỤM ĐẶC TRƯNG:")
for cid, feats in clustering_res["feature_clusters"].items():
    print(f"   • Cụm {cid} ({len(feats)} biến): {', '.join(feats[:4])}{'...' if len(feats) > 4 else ''}")

# 2. Phân tích Thành phần Chính (PCA: fit toàn bộ 32 components)
pca_res = analyzer.analyze_pca()

print("\n📈 PCA SUB-DIAGNOSTICS (32 Principal Components):")
print(f"   • PC1 giải thích:         {pca_res['explained_variance_ratio'][0]*100:.2f}% phương sai")
print(f"   • PC1-PC3 tích lũy:       {pca_res['cumulative_variance'][2]*100:.2f}% phương sai")
print(f"   • Số PC đạt >= 90%:       {next(i+1 for i, v in enumerate(pca_res['cumulative_variance']) if v >= 0.90)} components")

print("\n🎯 TOP PRINCIPAL COMPONENTS CÓ TƯƠNG QUAN CAO NHẤT VỚI LƯỢNG MƯA (|ρ_s|):")
print(pca_res["pc_target_association"].sort_values("abs_spearman_with_target", ascending=False).head(5).to_string(index=False))

# 3. Trực quan hóa Dendrogram & PCA Scree/Association
fig_c1 = viz.visualize_dendrogram(figsize=(14, 6), return_fig=True)
plt.show()

fig_c2 = viz.visualize_pca(top_k=10, figsize=(14, 6), return_fig=True)
plt.show()


🌳 PHÂN CỤM THỨ BẬC CÁC ĐẶC TRƯNG TUYẾN TÍNH (Average Linkage):
   • Thước đo khác biệt:     1 - |spearman_rho| (correlation-derived dissimilarity)
   • Phương pháp liên kết:    average
   • Số cụm yêu cầu (Heuristic): 5
   • Số cụm thực tế (Cut):   5

📦 DANH SÁCH CÁC CỤM ĐẶC TRƯNG:
   • Cụm 1 (8 biến): Tốc độ gió 2m, Tốc độ gió tối đa 2m, Tốc độ gió tối thiểu 2m, Biên độ gió 2m...
   • Cụm 2 (14 biến): Bức xạ sóng ngắn bề mặt, Bức xạ trực tiếp pháp tuyến, Bức xạ quang hợp tổng, Bức xạ UVA...
   • Cụm 3 (6 biến): Bức xạ sóng ngắn trời quang, Bức xạ đỉnh khí quyển, Bức xạ quang hợp trời quang, Bức xạ sóng dài xuống...
   • Cụm 4 (3 biến): Điểm sương 2m, Độ ẩm tuyệt đối 2m, Áp suất bề mặt
   • Cụm 5 (1 biến): Bức xạ khuếch tán

📈 PCA SUB-DIAGNOSTICS (32 Principal Components):
   • PC1 giải thích:         39.52% phương sai
   • PC1-PC3 tích lũy:       77.44% phương sai
   • Số PC đạt >= 90%:       6 components

🎯 TOP PRINCIPAL COMPONENTS CÓ TƯƠNG QUAN CAO NHẤT VỚI LƯỢNG MƯA (|ρ_s|):
  pc  

In [5]:
# 1. Phân rã Hướng gió Tuần hoàn (Circular Wind Decomposition Diagnostic - Supplementary)
wind_res = analyzer.analyze_circular_diagnostics()

print("🧭 CHẨN ĐOÁN PHÂN RÃ HƯỚNG GIÓ TUẦN HOÀN (SUPPLEMENTARY DIAGNOSTIC ONLY):")
print(f"   • Phương pháp: {wind_res['method_note']}")
print(f"   • Nguyên tắc:  {wind_res['policy']}")
df_wind = pd.DataFrame(wind_res["variables"])
print(df_wind.to_string(index=False))

# 2. Deep Dive Bức xạ Solar & UV
import seaborn as sns
rad_res = analyzer.analyze_radiation_uv_deep_dive()

print(f"\n☀️ DEEP DIVE CÁC KÊNH BỨC XẠ & UV ({len(rad_res['columns'])} kênh):")
df_rad_assoc = pd.DataFrame(rad_res["target_associations"]).sort_values("spearman_rho", ascending=False)
print("   • Tương quan với lượng mưa:")
print(df_rad_assoc.to_string(index=False))

# Trực quan hóa ma trận tương quan nội bộ nhóm bức xạ
fig_rad, ax_rad = plt.subplots(figsize=(10, 8))
df_rad_mat = pd.DataFrame(rad_res["within_group_spearman"])
mask_rad = np.triu(np.ones_like(df_rad_mat, dtype=bool))
sns.heatmap(df_rad_mat, mask=mask_rad, annot=True, fmt=".2f", cmap="YlOrRd", center=0.5, ax=ax_rad)
ax_rad.set_title("Solar & UV Radiation Cross-Correlation Matrix (Spearman ρ)", fontsize=12, fontweight="bold")
plt.show()


🧭 CHẨN ĐOÁN PHÂN RÃ HƯỚNG GIÓ TUẦN HOÀN (SUPPLEMENTARY DIAGNOSTIC ONLY):
   • Phương pháp: First-pass sin/cos decomposition. Supplementary only; not injected into primary VIF, PCA, or F0-F3 feature sets.
   • Nguyên tắc:  sin/cos not injected into primary VIF/PCA/clustering; not added to F0-F3 feature sets
 raw_variable  sin_spearman_with_target  cos_spearman_with_target
 Hướng gió 2m                   -0.4499                    0.0313
Hướng gió 10m                   -0.4582                    0.0282

☀️ DEEP DIVE CÁC KÊNH BỨC XẠ & UV (11 kênh):
   • Tương quan với lượng mưa:
                    feature  spearman_rho
      Bức xạ sóng dài xuống        0.4512
          Bức xạ khuếch tán        0.2439
Bức xạ quang hợp trời quang        0.2391
      Bức xạ đỉnh khí quyển        0.2284
Bức xạ sóng ngắn trời quang        0.1279
                 Bức xạ UVB       -0.3104
                  Chỉ số UV       -0.3366
                 Bức xạ UVA       -0.3867
      Bức xạ quang hợp tổng       -0.47

In [6]:
# Xuất Artifact Báo cáo Tương quan & Đa cộng tuyến (JSON)
OUTPUT_DIR = ROOT_DIR / "reports" / "eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

report_data = analyzer.generate_correlation_report()
artifact_path = OUTPUT_DIR / "01.3_correlation_report.json"
canonical_path = OUTPUT_DIR / "correlation_report.json"

for path in [artifact_path, canonical_path]:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(report_data, f, indent=2, ensure_ascii=False)

print(f"💾 Báo cáo tương quan & đa cộng tuyến xuất bản thành công tại:")
print(f"   • {artifact_path}")
print(f"   • {canonical_path}")
print("\n📊 TỔNG KẾT BẰNG CHỨNG (OBSERVED EVIDENCE):")
for obs in report_data["observed_evidence"]:
    print(f"   ✓ {obs}")
print("\n⚠️ RANH GIỚI KHÔNG PHẢI KẾT LUẬN TỪ EDA:")
print(f"   {report_data['not_conclusions_from_eda']}")


💾 Báo cáo tương quan & đa cộng tuyến xuất bản thành công tại:
   • D:\DS108_project\reports\eda\01.3_correlation_report.json
   • D:\DS108_project\reports\eda\correlation_report.json

📊 TỔNG KẾT BẰNG CHỨNG (OBSERVED EVIDENCE):
   ✓ Highest contemporaneous monotonic associations with rainfall on Dev Train occur among moisture and humidity predictors.
   ✓ Predictor space exhibits rank-deficiency (rank 29 < 32) and condition number exceeding 1e10, driven by deterministic definitions (thermal range, wind range).
   ✓ Hierarchical clustering (average linkage, 1-|rho_s|) organizes the 32 linear predictors into 5 redundancy clusters.
   ✓ Solar and UV radiation channels exhibit strong within-group cross-correlation (|rho_s| > 0.85).
   ✓ Wind direction circular decomposition reveals modest directional correlation signals, preserved as supplementary diagnostics.

⚠️ RANH GIỚI KHÔNG PHẢI KẾT LUẬN TỪ EDA:
   01.3 provides diagnostic association and collinearity evidence only on Dev Train. Featu